# Magnet Design Optimization for Magnetic Resonance Imaging - Linear Accelerator (MRI-LINAC) Radiation Therapy

**Team Members:** Vinicius Jameli, Felix Deng, Jie Mao

**Table of Contents:** 
- [Introduction](#introduction)
- [Problem formulation](#problem-formulation)
- [Baseline model](#baseline-model)
- [Improvement: Sweep line maximal clique constraints](#improvement-sweep-line-maximal-clique-constraints)
- [References](#references)

## Introduction

Modern magnetic resonance imaging (MRI) technology has been one of the most promising modalities in soft-tissue imaging.  Its fine definition of anatomical anomalies has been shown to be beneficial for not only disease diagnostics, but also for coupling with disease treatments such as radiation therapy to increase the precision of treatments [1]. 
However, MRI and other medical technologies are historically designed to function as separate devices, and these systems can often interfere with each other when coupled to function in close proximity. For example, the magnetic field introduced by the charged particles from the X-Ray beams would interfere the magnetic field generated by the MRI machine, disrupting the magnetic field homogeneity and significantly reducing imaging quality [1]. 
Therefore, the evolving needs in modern medicine demands a more flexible, yet general, approach to optimize MRI magnet design when additional design constraints are involved. 

MRI images are generated within a homogeneous magnetic field [2], often created by two coaxial located electromagnets (see Figure 1a) positioned in equidistance from the imaging volume or the volume of interest (VOI), where the magnetic gradients generated by the two magnets are cancelled out under perfect alignment [3]. 
In this project, we aim to extend the existing practices of optimizing magnet design using mixed linear programming to incorporate arbitrarily complex asymmetries [4]. 
More specifically, we are optimizing the spatial location of electromagnetic coils in non-coaxial, yet still concentric, configurations, as shown in Figure 1b. Through this approach, the optimized magnet design will still generate magnetic field remains homogeneous (within a small tolerance), while minimizing the overall weight of the coils to minimize cost. 

<center>
    <img src="./images/coaxial_mangets.jpg" width="80%">
    <p>Figure 1: Schematic representation of coaxial and non-coaxial magnet configurations.</p>
</center>

## Problem Formulation

In order to compute the resulting magnetic field generated by an electromagnetic coil in space, we first define a few variables, as illustrated in Figure 2: 

- Spatial coordinates: $(x, y, z)$
- Internal radius: $r$
- Cross section height: $h$
- Cross section width: $w$
- Number of wire windings: $N$
- Current: $i$

<center>
    <img src="./images/parameters.jpg" width="80%">
    <p>Figure 2. Parameters for magnetic coil and spatial location in space.</p>
</center>

However, modelling the magnetic field generated by an ideal loop of coil based on its geometry involves highly nonlinear relationship. Instead, we exploit the fact that the current $i$ has a linear relationship with the resultant magnetic field. 
Therefore, we only consider $i$ as a variable and fix all other variables, and we are hence solve a large-scale mix-integer programming problem instead of solving a highly nonlinear but small optimization problem. 

In this project, we set $N = 10^7$ and let $w = 1.4 h$. We can further reduce the number of variables by defining $\sigma$ as the area of the cross-section, where $\sigma = h \times w$. Then, we discretize all the spatial coordinates $(x, y, z)$, widths $w$, and internal radius $r$. Such that, we can pre-compute an incidence matrix $\bold{A}$ as an input to the optimization model, where $a_{mn}^s$ is the magnetic field intensity generated by a unit current at the $m$-th target point in imaging domain $D$ generated by the $n$-th coil in the coil domain $\mathcal{N}$ with the $s$-th cross section in a finite set $\mathcal{S}$ that indexes all potential cross-sections that coils may adopt [5], as illustrated in Figure 3. 

<center>
    <img src="./images/domain.jpg" width="75%">
    <p>Figure 3: Illustration of the optimization space in a cross-section of the magnetic coil, reproduced from Ref. [4].</p>
</center>

At the same time, we also make the following assumptions that are commonly used in the field of MRI technology: 

- We assume that the magnetic field is propagated in free space, and the coils are circular with rectangular cross-sections and are coaxial (i.e., share the same $z$-axis). Therefore, it is sufficient to constraint the magnetic field in a semi-circle in the $y=0$ plane due to symmetry, as shown in Figure 3. 
- The resultant magnetic field generated by a coil is linearly proportional to the current in the coil (i.e., linearity), and the resultant magnetic field generated by $N$ coils is given by the superposition of the magnetic fields generated by each of the $N$ coils. This assumption allows for a linear optimization model. 
- It is only necessary to constraint the axial component of the magnetic field ($B_z$) in order to guarantee homogeneity in the VOI, since the radial component ($B_r$ or $B_x$) becomes negligible through quadrature suppression when $B_z$ is homogeneous [3]. Therefore, we only need to constraint $M$ target points surrounding the surface boundary ($\partial D$) of the spherical domain $D$ (i.e., a set of target points $\mathcal{M}$). 
- In reality, the thickness of electrical coils is non-negligible, and we need to avoid overlapping coils in space. Let $\sigma^s$, $w^s$, and $h^s$ denote the area, width, and height of cross-section $s \in \mathcal{S}$. For each pair of candidate coil positions $n$ and $n'$, we define $\mathcal{S}_{nn'}$ as the set of overlapping cross-sections at these two coil locations. Cross-sections $s$ and $s'$ are considered overlapping at coil positions $n$ and $n'$ when $|{z_n - z_{n'}}| < \frac{h^s}{2} + \frac{h^{s'}}{2} + L_z$ and $|{r_n - r_{n'}}| < \frac{w^s}{2} + \frac{w^{s'}}{2} + L_r$, where $L_z$ and $L_r$ are the minimum required gaps between any two coils on the same pole along the $z$- and $r$-axis, respectively. 

Following the formulation originally developed by Dayarian et al. [4], the magnet design optimization problem can essentially be formulated as the following mix-integer linear programming (MILP) problem: 

$$
\begin{align}
\text{minimize} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} r_n \sigma^s x_{n}^{s} \\
\text{subject to} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \leq B_{0} (1 + \varepsilon), \quad \forall m \in \mathcal{M} \\
& \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \geq B_{0} (1 - \varepsilon), \quad \forall m \in \mathcal{M} \\
& |i_{n}^{s}| \leq J_{c} \sigma^{s} x_{n}^{s}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& \sum\limits_{s \in \mathcal{S}} x_{n}^{s} \leq 1, \quad \forall n \in \mathcal{N} \\
& x_{n}^{s} + x_{n'}^{s'} \leq 1, \quad \forall n, n' \in \mathcal{N}, \forall (s, s') \in \mathcal{S}_{nn'} \\
& i_{n}^{s} \text{ free}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& x_{n}^{s} \in \{0,1\}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} 
\end{align}
$$

Note that: 
- $i_n^s$ is a decision variable that takes the current associated with a thick coil at location $n$ in cross-section $s$. $i_n^s$ is used as a free variable in Eqn. 7, because it can take negative values for current passing through the reversed direction. 
- $x_n^s$ is a binary variable that takes the value of 1 if and only if coil $n$ in cross-section $s$ has a non-zero current, and it takes the value of 0 otherwise (Eqn. 8). 
- The presented MILP aims to minimize the total volume of superconductive material required for the magnet construction (Eqn. 1). 
- The constraints in Eqn. 2 and 3 enforce the magnetic field generated at each target point in the VOI is homogeneous with a magnetic intensity of $B_0$, where a small tolerance $\epsilon$ is allowed. 
- The constraint in Eqn. 4 ensures that the current density in active coils remains below the critical current density threshold $J_c$. 
- The constraint in Eqn. 5 ensures that each candidate coil location can take at most one cross-section, and the constraint in Eqn. 6 prevents coil overlapping by prohibiting imcompatible cross-sections from being simultaneously assigned to any pair of coil positions $n$ and $n'$ according to the predifined set $\mathcal{S}_{nn'}$ (the last assumption discussed above). 

In [1]:
import numpy as np 
from scipy import constants 
from typing import Tuple, Optional 

In [2]:
# Define constants for the problem 
B_0 = 0.5 # in Tesla, desired magnetic field strength 
EPSILON = 1e-5 # in Tesla, tolerance for magnetic field strength 
J_C = 2.1e8 # in A/m^2, critical current density 

In [3]:
# Setting the semi-circular imaging domain M on the x-z plane centered at origin 
VOI_RADIUS = 0.2 # in meters 
VOI_NUM_POINTS = 40 

angles = np.linspace(-np.pi / 2, np.pi / 2, VOI_NUM_POINTS) 
voi_boundary_points = np.zeros((VOI_NUM_POINTS, 3)) 
voi_boundary_points[:, 0] = VOI_RADIUS * np.cos(angles) # x-coordinates
voi_boundary_points[:, 1] = 0.0 # y-coordinates (all points lie on the x-z plane)
voi_boundary_points[:, 2] = VOI_RADIUS * np.sin(angles) # z-coordinates

In [4]:
# Setting default properties of the coil domain N

# Compute the coil domain geometries 
COIL_WIDTH_MIN, COIL_WIDTH_MAX = 0.005, 0.10 # in meters
COIL_NUM_WIDTHS = 20 # number of discrete widths to consider for the coil
CROSS_SECTION_RATIO = 1.4 # w = h * ratio 

coil_widths = np.linspace(COIL_WIDTH_MIN, COIL_WIDTH_MAX, COIL_NUM_WIDTHS)
coil_heights = coil_widths / CROSS_SECTION_RATIO
coil_cross_section_areas = coil_widths * coil_heights

# Compute coil positions in a grid pattern 
COIL_RADIUS_MIN, COIL_RADIUS_MAX = 0.10, 0.48 # in meters
COIL_Z_MIN, COIL_Z_MAX = 0.34, 0.42 # in meters
COIL_GRID_SPACING = 0.02 # in meters
Z_AXIS_X, Z_AXIS_Y = 0.0, 0.0 # x and y coordinates of the z-axis where coils are placed

coil_z_positions = [] 
z = COIL_Z_MIN 
while z <= COIL_Z_MAX + 1e-10: 
    coil_z_positions.append(z) 
    z += COIL_GRID_SPACING
coil_radius_values = [] 
r = COIL_RADIUS_MIN 
while r <= COIL_RADIUS_MAX + 1e-10: 
    coil_radius_values.append(r) 
    r += COIL_GRID_SPACING
    
coil_positions = [] 
coil_radii = []
for z in coil_z_positions: 
    for r in coil_radius_values: 
        coil_positions.append([Z_AXIS_X, Z_AXIS_Y, z]) # all combinations for the north half 
        coil_radii.append(r)
for z in coil_z_positions: 
    for r in coil_radius_values: 
        coil_positions.append([Z_AXIS_X, Z_AXIS_Y, -z]) # all combinations for the south half 
        coil_radii.append(r)
coil_positions = np.array(coil_positions)
coil_radii = np.array(coil_radii)

NUM_COILS = len(coil_positions)

# Other coil parameters
COIL_MIN_DISTANCE = 0.01 # in meters 
COIL_TURNS = 10000000 # number of wire turns 

In [5]:
from compute_amns import AMNSMatrix
from compute_amns import compute_amns 

In [ ]:
# Since the computation of the AMNS matrix is computationally resource intensive, 
# we directly load a pre-computed matrix from the repository for our experiments.

AMNS_MATRIX = compute_amns(
    opt_points=voi_boundary_points,
    coil_positions=coil_positions,
    coil_radii=coil_radii,
    widths=coil_widths,
    cross_section_ratio=CROSS_SECTION_RATIO,
    N=COIL_TURNS, 
    current=1.0, 
    order_n1=18, order_n2=10, tolerance=1e-4 
)

print("AMNS matrix computed with Bx in shape:", AMNS_MATRIX.Bx.shape)

AMNS matrix computed with Bx in shape: (40, 200, 20)


In [6]:
# Saving and loading the AMNS matrix using pickle 
def save_amns(amns: AMNSMatrix, filepath: str):
    filepath = str(filepath)
    if filepath.endswith(".npz"):
        np.savez(
            filepath,
            Bx=amns.Bx,
            By=amns.By,
            Bz=amns.Bz,
            num_opt_points=np.array(amns.num_opt_points),
            num_coils=np.array(amns.num_coils),
            num_widths=np.array(amns.num_widths),
        )
    else: 
        raise ValueError("Unsupported file format. Please use .npz for NumPy compressed format.")

def load_amns(filepath: str) -> AMNSMatrix:
    filepath = str(filepath)
    if filepath.endswith(".npz"):
        data = np.load(filepath)
        return AMNSMatrix(
            Bx=data["Bx"],
            By=data["By"],
            Bz=data["Bz"],
            num_opt_points=int(data["num_opt_points"]),
            num_coils=int(data["num_coils"]),
            num_widths=int(data["num_widths"]),
        )
    else: 
        raise ValueError("Unsupported file format. Please use .npz for NumPy compressed format.")


In [7]:
AMNS_MATRIX = load_amns("AMNS_semicircle_40.npz")
print("Loaded AMNS matrix with shape:", AMNS_MATRIX.Bx.shape)

Loaded AMNS matrix with shape: (40, 200, 20)


In [8]:
# Compute the S_nn' set to avoid coil overlapping 
# Find all (n1, s1, n2, s2) pairs that would overlap 
OVERLAP_PAIRS = [] 

def _would_overlap(n1: int, s1: int, n2: int, s2: int) -> bool:
    # Z overlap: centers within half-heights + min_distance
    z1 = coil_positions[n1, 2]
    z2 = coil_positions[n2, 2]
    h1 = coil_heights[s1]
    h2 = coil_heights[s2]
    z_overlap = abs(z1 - z2) < (h1 / 2 + h2 / 2 + COIL_MIN_DISTANCE)

    # Radius overlap: radii within half-widths + min_distance
    r1 = coil_radii[n1]
    r2 = coil_radii[n2]
    w1 = coil_widths[s1]
    w2 = coil_widths[s2]
    r_overlap = abs(r1 - r2) < (w1 / 2 + w2 / 2 + COIL_MIN_DISTANCE)

    return z_overlap and r_overlap


# North pole coils 
for n1 in range(NUM_COILS // 2): 
    for n2 in range(n1 + 1, NUM_COILS // 2):
        for s1 in range(COIL_NUM_WIDTHS):
            for s2 in range(COIL_NUM_WIDTHS):
                if _would_overlap(n1, s1, n2, s2):
                    OVERLAP_PAIRS.append((n1, s1, n2, s2))
# South pole coils 
for n1 in range(NUM_COILS // 2, NUM_COILS): 
    for n2 in range(n1 + 1, NUM_COILS):
        for s1 in range(COIL_NUM_WIDTHS):
            for s2 in range(COIL_NUM_WIDTHS):
                if _would_overlap(n1, s1, n2, s2):
                    OVERLAP_PAIRS.append((n1, s1, n2, s2))

## Baseline Model

For the baseline, we implemented the MILP problem modelled with Eqn. 1 to 8 using Gurobi. 

In [ ]:
import gurobipy as gp
from gurobipy import GRB

model_baseline = gp.Model("baseline_coil_optimization") 

# Gurobi solver configurations 
model_baseline.setParam('TimeLimit', 600) # seconds 
model_baseline.setParam('MIPGap', 0.01) # 1% optimality gap
model_baseline.setParam('FeasibilityTol', 1e-9)
model_baseline.setParam('IntFeasTol', 1e-8)
model_baseline.setParam('NumericFocus', 1) # 0=auto, 1=careful, 2=aggressive, 3=very aggressive 
model_baseline.setParam('Threads', 1) 
model_baseline.setParam('OutputFlag', 1) # 0=silent, 1=normal

# Create decision variables 
# i[n, s]: Current through coil n with width s (continuous) [Eqn. 7]
vars_i = model_baseline.addVars(
    NUM_COILS,
    COIL_NUM_WIDTHS,
    lb=-GRB.INFINITY,
    ub=GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="i",
)
# x[n, s]: Binary indicator if coil n uses width s [Eqn. 8]
vars_x = model_baseline.addVars(
    NUM_COILS,
    COIL_NUM_WIDTHS,
    vtype=GRB.BINARY,
    name="x",
)

# Convert to 2D arrays for easier indexing
array_i = np.array(
    [[vars_i[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(NUM_COILS)],
    dtype=object,
)
array_x = np.array(
    [[vars_x[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(NUM_COILS)],
    dtype=object,
)

# Set objective: minimize total "coil volume" (radius * area) [Eqn. 1]
# This approximates minimizing total current 
obj = gp.LinExpr(
    (
        coil_radii[:, None] * coil_cross_section_areas[None, :]).flatten().tolist(), 
        list(array_x.flatten()
    )
)
   
model_baseline.setObjective(obj, GRB.MINIMIZE)

Set parameter TimeLimit to value 600
Set parameter MIPGap to value 0.01
Set parameter FeasibilityTol to value 1e-09
Set parameter IntFeasTol to value 1e-08
Set parameter NumericFocus to value 1
Set parameter Threads to value 1
Set parameter OutputFlag to value 1


In [17]:
# Add homogeneity constraints on Bz field [Eqn. 2 and 3]
upper_bound = B_0 * (1 + EPSILON)
lower_bound = B_0 * (1 - EPSILON)

if AMNS_MATRIX.Bz.ndim == 3: 
    bz_2d = AMNS_MATRIX.Bz.reshape(VOI_NUM_POINTS, NUM_COILS * COIL_NUM_WIDTHS)
else: 
    bz_2d = AMNS_MATRIX.Bz # already (num_opt_points, num_coils * num_widths)

i_flat = list(array_i.flatten())

for m in range(VOI_NUM_POINTS):
    # Build field expression: sum(AMNS[2,m,n,s] * x[n,s])
    # Compute Bz at point m as sum over coils and widths
    expr = gp.LinExpr(bz_2d[m].tolist(), i_flat)
    
    # Add constraints: lower_bound <= Bz <= upper_bound
    model_baseline.addConstr(expr >= lower_bound, name=f"homo_lower_{m}")
    model_baseline.addConstr(expr <= upper_bound, name=f"homo_upper_{m}")

In [18]:
# Add symmetry constraint between north and south pole coils 
half_num_coils = NUM_COILS // 2
for n in range(half_num_coils):
    for s in range(COIL_NUM_WIDTHS):
        # North coil n pairs with south coil n + half
        model_baseline.addConstr(
            array_x[n, s] == array_x[n + half_num_coils, s],
            name=f"symmetry_{n}_{s}"
        )

In [19]:
# Add upper bound constraint on current density [Eqn. 4]
for n in range(NUM_COILS): 
    for s in range(COIL_NUM_WIDTHS): 
        # I_max = J_crit * Area / N
        i_max = J_C * coil_cross_section_areas[s] / COIL_TURNS 
        model_baseline.addConstr(array_i[n, s] <= i_max * array_x[n, s], name=f"current_upper_{n}_{s}")
        model_baseline.addConstr(array_i[n, s] >= -i_max * array_x[n, s], name=f"current_lower_{n}_{s}")

In [20]:
# Add constraint such that each coil position uses at most one width [Eqn. 5]
for n in range(NUM_COILS): 
    expr = gp.LinExpr() 
    for s in range(COIL_NUM_WIDTHS): 
        expr += array_x[n, s]
    model_baseline.addConstr(expr <= 1, name=f"unique_width_{n}")

In [21]:
# Add constraint to prevent overlapping coils [Eqn. 6]
for n1, s1, n2, s2 in OVERLAP_PAIRS: 
    # Direct constraint: y[n1, s1] + y[n2, s2] <= 1
    model_baseline.addConstr(
        array_x[n1, s1] + array_x[n2, s2] <= 1, 
        name=f"overlap_{n1}_{s1}_{n2}_{s2}"
    )
    # Cross constraint: y[n1, s2] + y[n2, s1] <= 1
    # Only add if different widths
    if s1 != s2:
        model_baseline.addConstr(
            array_x[n1, s2] + array_x[n2, s1] <= 1,
            name=f"overlap_cross_{n1}_{s2}_{n2}_{s1}",
        )

In [22]:
# Baseline model in solving the problem 
import time 

model_baseline.update()

start_time = time.time()
model_baseline.optimize() 
solve_time = time.time() - start_time

print("Solve time (seconds):", solve_time)
print("Optimization status:", model_baseline.Status)
print("Objective value:", model_baseline.ObjVal if model_baseline.SolCount > 0 else float("inf"))

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
FeasibilityTol  1e-09
IntFeasTol  1e-08
MIPGap  0.01
NumericFocus  1
Threads  1

Optimize a model with 1611264 rows, 8000 columns and 3545968 nonzeros (Min)
Model fingerprint: 0x7ab96581
Model has 4000 linear objective coefficients
Variable types: 4000 continuous, 4000 integer (4000 binary)
Coefficient statistics:
  Matrix range     [4e-04, 2e+01]
  Objective range  [2e-06, 3e-03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-01, 1e+00]
Presolve removed 1586247 rows and 2000 columns
Presolve time: 1.56s
Presolved: 25017 rows, 6040 columns, 388319 nonzeros
Variable types: 4040 continuous, 2000 integer (2000 binary)
Performing another presolve...
Presolve removed 1773 rows and 0 columns
Presolve time: 1.28s

Root simplex log...

Iteration    Objective       P

## Improvement: Sweep Line Maximal Clique Constraints 

To improve the baseline model, 

## References

1. G. P. Liney, B. Whelan, B. Oborn, M. Barton, and P. Keall, “MRI-Linear Accelerator Radiotherapy Systems,” Clinical Oncology, vol. 30, no. 11, pp. 686–691, Nov. 2018, doi: 10.1016/j.clon.2018.08.003.
1. A. Sattarov, P. McIntyre, and L. Motowidlo, “High-Field Open MRI for Breast Cancer Screening,” IEEE Trans. Appl. Supercond., vol. 25, no. 3, pp. 1–5, Jun. 2015, doi: 10.1109/TASC.2014.2377049.
1. Hao Xu, S. M. Conolly, G. C. Scott, and A. Macovski, “Homogeneous magnet design using linear programming,” IEEE Trans. Magn., vol. 36, no. 2, pp. 476–483, Mar. 2000, doi: 10.1109/20.825817.
1. I. Dayarian, T. C. Y. Chan, D. Jaffray, and T. Stanescu, “A mixed-integer optimization approach for homogeneous magnet design,” Technology, vol. 06, no. 02, pp. 49–58, Jun. 2018, doi: 10.1142/S2339547818500036.
1. L. K. Forbes, S. Crozier, and D. M. Doddrell, “Rapid computation of static fields produced by thick circular solenoids,” IEEE Trans. Magn., vol. 33, no. 5, pp. 4405–4410, Sep. 1997, doi: 10.1109/20.620453.